In [0]:
# ============================================================
# 01_procurement_gold  —  Stage 1 of the SAP BDC Multi-Agent
#   Finance Intelligence use case
#
# Purpose: build the PROCUREMENT gold layer from SAP BDC
#   vendor-performance + supplier Delta Shares.
#   These gold tables become the foundation for the
#   Vendor Performance sub-agent's tools.
#
# Companion to: gold_monthly_cashflow (Phase 1 cash-flow gold)
# ============================================================

VP = "`bdc_share_vendorperformance`.`s4_zvendorperformance_dp_srv:v1`.`s4custom_vendorperformance`"
SUP = "`bdc_share_supplier`.`supplier`.`supplier`"

# defensive re-check: confirm the source tables are reachable and sized as expected
print("Vendor performance rows:",
      spark.sql(f"SELECT COUNT(*) FROM {VP}").collect()[0][0])
print("Supplier rows:",
      spark.sql(f"SELECT COUNT(*) FROM {SUP}").collect()[0][0])

In [0]:
# ============================================================
# Cell 2 — build gold_vendor_performance
#   One row per vendor: OTIF, on-time, in-full, rejection rates,
#   cycle time, order counts and value. String KPI columns
#   ("true"/"false") become real rates.
# ============================================================
spark.sql("""
CREATE OR REPLACE TABLE workspace.default.gold_vendor_performance AS
SELECT
    VendorAccountNumber_LIFNR                         AS vendor_id,
    MAX(NAME1)                                        AS vendor_name,
    MAX(CountryKey_LAND1)                             AS country,
    COUNT(*)                                          AS total_po_lines,
    ROUND(AVG(CASE WHEN VendorOnTimeDelivery       = 'true' THEN 1.0 ELSE 0.0 END), 4) AS on_time_rate,
    ROUND(AVG(CASE WHEN VendorInFullDelivery       = 'true' THEN 1.0 ELSE 0.0 END), 4) AS in_full_rate,
    ROUND(AVG(CASE WHEN VendorOnTimeInFullDelivery = 'true' THEN 1.0 ELSE 0.0 END), 4) AS otif_rate,
    ROUND(AVG(CASE WHEN VendorInvoiceAccuracy      = 'true' THEN 1.0 ELSE 0.0 END), 4) AS invoice_accuracy_rate,
    ROUND(AVG(CASE WHEN IsRejected                 = 'true' THEN 1.0 ELSE 0.0 END), 4) AS rejection_rate,
    ROUND(AVG(VendorCycleTimeInDays), 2)              AS avg_cycle_time_days,
    MAX(VendorCycleTimeInDays)                        AS max_cycle_time_days,
    SUM(CASE WHEN VendorOnTimeInFullDelivery = 'false' THEN 1 ELSE 0 END) AS otif_fail_count,
    ROUND(SUM(NetOrderValueinTargetCurrency_NETWR), 2)  AS total_order_value,
    MAX(TargetCurrency_TCURR)                         AS currency
FROM bdc_share_vendorperformance.`s4_zvendorperformance_dp_srv:v1`.s4custom_vendorperformance
WHERE VendorAccountNumber_LIFNR IS NOT NULL
GROUP BY VendorAccountNumber_LIFNR
""")
print("gold_vendor_performance created.")
spark.sql("SELECT vendor_id, vendor_name, total_po_lines, otif_rate, on_time_rate, rejection_rate, avg_cycle_time_days, total_order_value, currency FROM workspace.default.gold_vendor_performance ORDER BY total_po_lines DESC").show(25, truncate=False)

In [0]:
# ============================================================
# Cell 3 — FINDING: supplier master cannot join to vendor performance
#   The vendor-performance feed uses vendor IDs V001-V021
#   (bakehouse demo vendors). The supplier master table uses
#   SAP 10-digit IDs (0000000001 ...) - a separate, unrelated
#   dataset. They share no join key, so a gold_vendor_master
#   table is not built. gold_vendor_performance stands alone
#   as the procurement gold layer.
# ============================================================
spark.sql("DROP TABLE IF EXISTS workspace.default.gold_vendor_master")
print("Confirmed: procurement gold layer = gold_vendor_performance only.")

In [0]:
# ============================================================
# Cell 4 — metadata / governance for gold_vendor_performance
# ============================================================
GVP = "workspace.default.gold_vendor_performance"
spark.sql(f"""
COMMENT ON TABLE {GVP} IS
'Vendor performance gold table for the SAP BDC procurement use case. One row per vendor, aggregated from the SAP vendor-performance data product. Holds OTIF (on-time-in-full) and related delivery quality rates, cycle time, rejection rate, order counts and order value. Use for vendor performance analysis, scorecards, and ranking questions.'
""")
col_comments = {
    "vendor_id":             "Vendor identifier (e.g. V001 to V021).",
    "vendor_name":           "Vendor or supplier name.",
    "country":               "Vendor country key.",
    "total_po_lines":        "Number of purchase-order lines for this vendor. A low count means the vendor rates are based on little data and should be treated with caution.",
    "on_time_rate":          "Fraction of PO lines delivered on time (0 to 1).",
    "in_full_rate":          "Fraction of PO lines delivered in full (0 to 1).",
    "otif_rate":             "On-Time-In-Full rate (0 to 1), the headline vendor performance metric.",
    "invoice_accuracy_rate": "Fraction of PO lines with an accurate invoice (0 to 1).",
    "rejection_rate":        "Fraction of PO lines that were rejected (0 to 1). Higher is worse.",
    "avg_cycle_time_days":   "Average vendor cycle time in days, order to delivery.",
    "max_cycle_time_days":   "Maximum vendor cycle time in days observed.",
    "otif_fail_count":       "Raw count of PO lines that failed OTIF (late and/or short).",
    "total_order_value":     "Total net order value across all PO lines, in the target currency.",
    "currency":              "Currency of total_order_value.",
}
for col, desc in col_comments.items():
    spark.sql(f"COMMENT ON COLUMN {GVP}.{col} IS '{desc}'")
print(f"Metadata applied: {len(col_comments)} column comments + 1 table comment.")
spark.sql(f"DESCRIBE TABLE {GVP}").show(30, truncate=False)